# 实验四：Kernel侧开发与性能分析——Ascend C基础开发与真实NPU性能采样

## 小节概述

基于 Ascend C 理解 Kernel 侧 CopyIn、Compute、CopyOut 三阶段，并完成基础功能验证和性能观察。

## 教程具体内容

以下内容围绕实验背景、任务准备、关键步骤和实验总结展开。


建议学时：2学时

## 实验任务

### 任务描述

---

原实验设计：

在华为云沙箱完成Kernel开发、错误注入、功能验证和性能采样。

修改后的设计：

本实验分为两个阶段：

- 阶段一（CANNLAB）：学习Ascend C语法，完成Kernel基础代码编写

- 阶段二（华为云沙箱）：在真实NPU环境进行功能验证和性能采样

修改原因：

Kernel代码编写需要反复调试和迭代，CANNLAB提供便捷的开发环境。性能采样和验证需要真实NPU硬件，在华为云沙箱完成更准确。

---

### 学习目标

小组分工：

- 开源软件开发工程师：在CANNLAB完成Kernel代码编写，在华为云沙箱完成性能测试

- 开源合规与安全工程师：使用msSanitizer进行内存安全审计

- 开源社区布道师：记录开发过程和性能分析结果

学习资源路径：

- CANNLAB平台：用于Kernel代码开发和初步测试

- 华为云沙箱：用于真实NPU验证和性能采样

- Ascend C文档：CANN自定义算子开发手册

- 性能分析工具：msProf使用指南

---

## 任务准备

### 前置知识

本实验需要提前学习以下相关知识：

- Ascend C编程模型与Kernel开发基础
- Python编程基础
- Linux命令行操作基础

### 实验环境准备

本实验在 **CANNLAB 算子开发环境** 中完成。环境需提供 Ascend NPU、CANN 9.0.0 或兼容版本、PyTorch/torch_npu、CMake、C++ 编译器和 msProf。


## 任务实施

### 实验要点

- 步骤一：理解 Ascend C 编程模型
- 步骤二：获取仓库内置的 AddCustomTemplate 完整工程
- 步骤三：理解 CopyIn、Compute、CopyOut 三阶段
- 步骤四：编译、部署并运行完整样例
- 步骤五：用 `torch.add` 验证 NPU 加法基准链路
- 步骤六：用 msProf 采集该实际 Python 程序
- 步骤七：读取真实 `op_summary_*.csv` 并形成分析结论


### 关键步骤

阶段一：CANNLAB中的Ascend C基础开发（建议用时：60分钟）

步骤一：理解Ascend C的编程模型

学习Ascend C的核心概念：

## Ascend C编程模型

### 核心概念

1. **TPipe**: 流水线管理器，控制数据在各级存储之间搬运

2. **TQue**: 队列管理器，管理Local Memory中的数据缓冲

3. **GlobalTensor**: 外部全局内存（DDR）的张量表示

4. **LocalTensor**: 内部局部内存（Buffer）的张量表示

### 数据流三阶段

1. **CopyIn**: 从Global Memory搬运数据到Local Memory

2. **Compute**: 在Local Memory中执行计算

3. **CopyOut**: 将计算结果从Local Memory搬回Global Memory

### 内存层次

- Global Memory（DDR）: 大容量外部显存，访问延迟高

- Local Memory（Buffer）: 芯片内部高速缓存，访问延迟低

- 通过流水线机制隐藏内存访问延迟

步骤二：获取仓库内置的 AddCustomTemplate 完整工程

课程仓库已经包含经过维护的 AddCustomTemplate 样例。下面把该工程复制到本实验的工作目录，并展示真实 Kernel 源码；后续编译、测试与源码使用同一个工程，避免示例代码和运行目标不一致：


In [ ]:
from pathlib import Path
import shutil

sample_relative = Path(
    "reference_practice/pytorch_online_inference_operator_optimize/"
    "src/add_custom_template"
)
search_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
source_project = next(
    (root / sample_relative for root in search_roots if (root / sample_relative).is_dir()),
    None,
)
if source_project is None:
    raise FileNotFoundError(
        "未找到仓库内置 AddCustomTemplate 工程；请从 cann-learning-hub 仓库根目录打开本 Notebook。"
    )

work_root = Path(".cann_course_work/01_04")
work_project = work_root / "add_custom_template"
work_root.mkdir(parents=True, exist_ok=True)
shutil.rmtree(work_project, ignore_errors=True)
shutil.copytree(source_project, work_project)

kernel_files = sorted((work_project / "op_kernel").glob("*.cpp"))
if not kernel_files:
    raise RuntimeError("AddCustomTemplate 工程中未找到 op_kernel/*.cpp。")

kernel_file = kernel_files[0]
print(f"样例来源: {source_project}")
print(f"实验副本: {work_project.resolve()}")
print(f"Kernel 文件: {kernel_file}")
print("\n=== Kernel 源码前 100 行 ===")
print("\n".join(kernel_file.read_text(encoding="utf-8").splitlines()[:100]))


步骤三：理解CopyIn、Compute、CopyOut三阶段

## 数据流三阶段详解

### CopyIn阶段

- **职责**: 从DDR搬运数据到Buffer

- **关键操作**: DataCopy、EnQue

- **性能考虑**: 尽量合并多次小规模搬运

### Compute阶段

- **职责**: 在Buffer中执行计算

- **关键操作**: Add、Mul等计算原语

- **性能考虑**: 利用AI Core的并行计算能力

### CopyOut阶段

- **职责**: 将结果从Buffer搬回DDR

- **关键操作**: DeQue、DataCopy、FreeTensor

- **性能考虑**: 及时释放Buffer供下次使用

### 流水线优势

- 多个block可以并行执行不同阶段

- 隐藏内存访问延迟

- 提高AI Core利用率

步骤四：编译、部署并运行完整样例

下面执行同一工程自带的 `run_test.sh`。脚本的编译、打包、部署或测试任一步骤失败，Notebook 单元都会直接失败，不再用后续 `echo` 覆盖真实退出码：


In [ ]:
%%bash
set -euo pipefail

PROJECT_DIR=".cann_course_work/01_04/add_custom_template"
if [[ ! -d "${PROJECT_DIR}" ]]; then
  echo "ERROR: 未找到 ${PROJECT_DIR}，请先运行上一个代码单元。" >&2
  exit 1
fi

cd "${PROJECT_DIR}"
bash run_test.sh
echo "AddCustomTemplate 编译、部署与测试: PASSED"


步骤五：用 `torch.add` 验证 NPU 加法基准链路

上一单元验证了仓库内置自定义算子工程。本单元使用 PyTorch 公共接口 `torch.add` 验证同一加法语义在当前 NPU 上可执行，并与 NumPy 基准比较。这里不把框架内置算子冒充为自定义算子：


In [ ]:
import numpy as np
import torch
import torch_npu

if not torch.npu.is_available():
    raise RuntimeError("当前 Python 会话无法访问 Ascend NPU。")

rng = np.random.default_rng(2026)
input_size = 1024
x_cpu = rng.standard_normal(input_size).astype(np.float32)
y_cpu = rng.standard_normal(input_size).astype(np.float32)

x_npu = torch.from_numpy(x_cpu).npu()
y_npu = torch.from_numpy(y_cpu).npu()
z_npu = torch.add(x_npu, y_npu)
torch.npu.synchronize()
z_cpu = z_npu.cpu().numpy()

baseline = x_cpu + y_cpu
max_error = float(np.max(np.abs(z_cpu - baseline)))
if not np.allclose(z_cpu, baseline, rtol=1e-5, atol=1e-5):
    raise AssertionError(f"NPU 与 NumPy 基准不一致，最大绝对误差: {max_error}")

print(f"最大绝对误差: {max_error:.8e}")
print("torch.add NPU 功能验证: PASSED")


步骤六：使用 msProf 采集实际程序

下面先生成一个真实执行 `torch.add` 的 Python 程序，再让 msProf 启动该程序。采集目标与功能验证目标保持一致：


In [ ]:
from pathlib import Path
import shlex
import shutil
import subprocess
import sys

profile_work = Path(".cann_course_work/01_04").resolve()
profile_work.mkdir(parents=True, exist_ok=True)
runner = profile_work / "profile_torch_add.py"
profile_dir = profile_work / "prof_result"
shutil.rmtree(profile_dir, ignore_errors=True)

runner.write_text(
    """import torch
import torch_npu

if not torch.npu.is_available():
    raise RuntimeError("Ascend NPU 不可用")

x = torch.randn(1024 * 1024, dtype=torch.float32, device="npu:0")
y = torch.randn(1024 * 1024, dtype=torch.float32, device="npu:0")
for _ in range(10):
    z = torch.add(x, y)
torch.npu.synchronize()
for _ in range(100):
    z = torch.add(x, y)
torch.npu.synchronize()
print("profile target checksum:", float(z.float().sum().cpu()))
""",
    encoding="utf-8",
)

msprof = shutil.which("msprof")
if not msprof:
    raise RuntimeError("当前环境未提供 msprof，请切换到包含性能分析工具的 CANNLab 镜像。")

application = f"{shlex.quote(sys.executable)} {shlex.quote(str(runner))}"
command = [
    msprof,
    f"--application={application}",
    f"--output={profile_dir}",
    "--task-time=on",
]
print("执行:", " ".join(shlex.quote(part) for part in command))
subprocess.run(command, check=True)

summary_files = sorted(profile_dir.rglob("op_summary_*.csv"))
if not summary_files:
    raise RuntimeError(f"msProf 已结束，但未在 {profile_dir} 下生成 op_summary_*.csv。")

print("msProf 采集: PASSED")
for summary_file in summary_files:
    print(f"  {summary_file}")


步骤七：读取真实 `op_summary_*.csv` 并形成分析结论

不同 CANN 版本导出的列可能略有差异。下面从实际 CSV 表头识别算子名称列和耗时列，不使用预设的 `summary.txt` 或虚构指标：


In [ ]:
import csv
from statistics import mean

summary_files = sorted(profile_dir.rglob("op_summary_*.csv"))
if not summary_files:
    raise RuntimeError("未找到 op_summary_*.csv，请先完成 msProf 采集。")

summary_file = summary_files[0]
with summary_file.open(encoding="utf-8-sig", newline="") as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)
    columns = reader.fieldnames or []

print(f"读取: {summary_file}")
print("CSV 列:", columns)
if not rows:
    raise RuntimeError("op_summary CSV 中没有性能记录。")

duration_candidates = ["Task Duration(us)", "Duration(us)"]
duration_column = next((name for name in duration_candidates if name in columns), None)
if duration_column is None:
    duration_column = next(
        (name for name in columns if "duration" in name.lower() and "us" in name.lower()),
        None,
    )
if duration_column is None:
    raise RuntimeError(f"无法从实际表头识别耗时列: {columns}")

name_candidates = ["Op Name", "Name", "OP Type", "Op Type", "Task Type"]
name_column = next((name for name in name_candidates if name in columns), None)

def parse_duration(row):
    try:
        return float((row.get(duration_column) or "").replace(",", ""))
    except ValueError:
        return None

add_rows = []
if name_column:
    add_rows = [row for row in rows if "add" in (row.get(name_column) or "").lower()]
selected_rows = add_rows or rows
durations = [value for row in selected_rows if (value := parse_duration(row)) is not None]
if not durations:
    raise RuntimeError(f"列 {duration_column} 中没有可解析的数值。")

scope = "名称包含 Add 的记录" if add_rows else "全部记录（实际表头未提供可识别的 Add 名称）"
print(f"分析范围: {scope}")
print(f"记录数: {len(durations)}")
print(f"平均 {duration_column}: {mean(durations):.6f}")
print(f"最大 {duration_column}: {max(durations):.6f}")
print(f"最小 {duration_column}: {min(durations):.6f}")
print("结论：以上数值来自本次 msProf 实测；仅凭耗时不能推断 AI Core 或带宽利用率。")


## 任务拓展

《Ascend C算子核心实现与性能分析报告.md》应包含：仓库内置 AddCustomTemplate 的来源路径，CopyIn、Compute、CopyOut 数据流说明，`run_test.sh` 的实际编译与测试结果，`torch.add` 与 NumPy 基准的误差，以及本次 msProf 生成的 `op_summary_*.csv` 路径、真实表头和耗时统计。若要判断 Compute-bound 或 Memory-bound，还需另外采集对应硬件指标，不能根据模拟值下结论。


## 实验总结

通过本次实验，完成了以下关键学习目标：

- 从仓库内置完整工程观察 Ascend C 的 CopyIn、Compute、CopyOut 三阶段；
- 编译、部署并运行 AddCustomTemplate 官方样例；
- 使用 `torch.add` 完成 NPU 功能基准验证；
- 使用 msProf 采集真实程序并读取 `op_summary_*.csv`；
- 区分实测耗时与需要额外硬件指标支持的性能瓶颈判断。


## 课后实践

请说明 Kernel 三阶段数据流，提交功能验证记录和一次性能采样分析。

## 参考答案

运行下面的代码单元查看参考分析。性能数据应填写实际环境中的采样结果。


In [ ]:
!cat ./answer/01.04_answer.md